In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

# Read image in grayscale
folder = os.path.dirname(os.path.abspath(_file_))
image_path = os.path.join(folder, "image.png")

image = cv2.imread(image_path, 0)

if image is None:
    print("Error: image.png not found!")
    exit()

# Step 1: Calculate histogram
histogram = np.zeros(256, dtype=int)

for pixel in image.flatten():
    histogram[pixel] += 1

total_pixels = image.size

# Step 2: Calculate total weighted sum
total_sum = 0

for i in range(256):
    total_sum += i * histogram[i]

# Step 3: Find optimal threshold manually
background_weight = 0
background_sum = 0
max_variance = 0
threshold_value = 0

for t in range(256):
    background_weight += histogram[t]

    if background_weight == 0:
        continue

    foreground_weight = total_pixels - background_weight

    if foreground_weight == 0:
        break

    background_sum += t * histogram[t]

    mean_background = background_sum / background_weight
    mean_foreground = (
        total_sum - background_sum
    ) / foreground_weight

    variance = (
        background_weight * foreground_weight
        * (mean_background - mean_foreground) ** 2
    )

    if variance > max_variance:
        max_variance = variance
        threshold_value = t

# Step 4: Apply manual thresholding
result = np.zeros_like(image)

for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        if image[i, j] > threshold_value:
            result[i, j] = 255
        else:
            result[i, j] = 0

print("Manual OTSU Threshold Value:", threshold_value)

# Step 5: Verify using OpenCV
cv_threshold, cv_result = cv2.threshold(
    image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("OpenCV OTSU Threshold Value:", cv_threshold)
print("Thresholds Match:", threshold_value == cv_threshold)
print("Images Match:", np.array_equal(result, cv_result))

# Step 6: Display results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(image, cmap="gray")
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(result, cmap="gray")
plt.title("Manual OTSU")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(cv_result, cmap="gray")
plt.title("OpenCV OTSU (Verification)")
plt.axis("off")

plt.tight_layout()
plt.show()